# 🔬 Hypothesis Testing Notebook

---

## Objective
Statistically test the patterns observed in the Data Analysis notebook to determine if they are significant or due to random chance.

## Understanding P-Values

| P-value | Meaning | Decision |
|---------|---------|----------|
| < 0.05 | Very unlikely by chance | ✅ Significant |
| ≥ 0.05 | Could be by chance | ❌ Not significant |

## Our Hypotheses

1. **H1:** Do different countries have significantly different inflation rates?
2. **H2:** Is price volatility related to inflation?
3. **H3:** Does inflation vary by month (seasonal patterns)?
4. **H4:** Have food prices significantly increased over time?

---
## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import kruskal, spearmanr, mannwhitneyu
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.style.use('seaborn-v0_8-whitegrid')
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded!")

---
## Step 2: Load Data

In [ ]:
df = pd.read_csv('../data/cleaned/food_prices_cleaned.csv', parse_dates=['date'])
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/reports', exist_ok=True)

print("✅ Data loaded!")
print(f"📊 {df.shape[0]:,} records from {df['country'].nunique()} countries")

---
## Hypothesis 1: Regional Differences in Inflation

**Question:** Do different countries have significantly different inflation rates?

**Test:** Kruskal-Wallis H Test (compares multiple groups)

In [ ]:
print("="*60)
print("🔬 H1: Do countries have different inflation rates?")
print("="*60)

country_groups = [group['inflation'].dropna().values for name, group in df.groupby('country')]
h1_stat, h1_p = kruskal(*country_groups)

print(f"\nTest: Kruskal-Wallis")
print(f"Statistic: {h1_stat:.2f}")
print(f"p-value: {h1_p:.2e}")
print()
if h1_p < 0.05:
    print("✅ SIGNIFICANT: Countries DO have different inflation rates!")
else:
    print("❌ Not significant: Differences might be random.")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
order = df.groupby('country')['inflation'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='country', y='inflation', order=order, ax=ax, palette='viridis')
ax.set_title(f'H1: Inflation by Country (p={h1_p:.2e})', fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.axhline(y=0, color='red', linestyle='--')
result = '✅ SIGNIFICANT' if h1_p < 0.05 else '❌ NOT SIGNIFICANT'
ax.text(0.02, 0.98, result, transform=ax.transAxes, fontweight='bold', va='top',
        bbox=dict(boxstyle='round', facecolor='white'))
plt.tight_layout()
plt.savefig('../outputs/figures/h1_regional_inflation.png', dpi=300, bbox_inches='tight')
plt.show()

---
## Hypothesis 2: Volatility and Inflation Relationship

**Question:** Is there a relationship between price volatility and inflation?

**Test:** Spearman Correlation

In [ ]:
print("="*60)
print("🔬 H2: Is volatility related to inflation?")
print("="*60)

valid = df[['price_range', 'inflation']].dropna()
h2_corr, h2_p = spearmanr(valid['price_range'], valid['inflation'])

print(f"\nTest: Spearman Correlation")
print(f"Correlation: {h2_corr:.4f}")
print(f"p-value: {h2_p:.2e}")
print()
if h2_p < 0.05:
    direction = "positive" if h2_corr > 0 else "negative"
    print(f"✅ SIGNIFICANT: There IS a {direction} relationship!")
else:
    print("❌ Not significant: No clear relationship.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(valid['price_range'], valid['inflation'], alpha=0.3, s=10)
z = np.polyfit(valid['price_range'], valid['inflation'], 1)
p = np.poly1d(z)
x_line = np.linspace(valid['price_range'].min(), valid['price_range'].max(), 100)
ax.plot(x_line, p(x_line), color='red', linewidth=2, label='Trend')
ax.set_title(f'H2: Volatility vs Inflation (r={h2_corr:.4f}, p={h2_p:.2e})', fontweight='bold')
ax.set_xlabel('Price Volatility')
ax.set_ylabel('Inflation (%)')
ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
result = '✅ SIGNIFICANT' if h2_p < 0.05 else '❌ NOT SIGNIFICANT'
ax.text(0.02, 0.98, result, transform=ax.transAxes, fontweight='bold', va='top',
        bbox=dict(boxstyle='round', facecolor='white'))
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/h2_volatility_inflation.png', dpi=300, bbox_inches='tight')
plt.show()

---
## Hypothesis 3: Seasonal Patterns

**Question:** Does inflation vary by month?

**Test:** Kruskal-Wallis H Test (comparing 12 months)

In [ ]:
print("="*60)
print("🔬 H3: Does inflation vary by month?")
print("="*60)

month_groups = [group['inflation'].dropna().values for name, group in df.groupby('month')]
h3_stat, h3_p = kruskal(*month_groups)

print(f"\nTest: Kruskal-Wallis")
print(f"Statistic: {h3_stat:.2f}")
print(f"p-value: {h3_p:.2e}")
print()
if h3_p < 0.05:
    print("✅ SIGNIFICANT: There IS a seasonal pattern!")
else:
    print("❌ Not significant: No clear seasonal pattern.")

In [ ]:
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=df, x='month', y='inflation', ax=ax, palette='coolwarm')
ax.set_xticklabels(months)
ax.set_title(f'H3: Inflation by Month (p={h3_p:.2e})', fontweight='bold')
ax.axhline(y=0, color='red', linestyle='--')
result = '✅ SIGNIFICANT' if h3_p < 0.05 else '❌ NOT SIGNIFICANT'
ax.text(0.02, 0.98, result, transform=ax.transAxes, fontweight='bold', va='top',
        bbox=dict(boxstyle='round', facecolor='white'))
plt.tight_layout()
plt.savefig('../outputs/figures/h3_seasonal_inflation.png', dpi=300, bbox_inches='tight')
plt.show()

---
## Hypothesis 4: Price Trend Over Time

**Question:** Have prices increased from early years to recent years?

**Test:** Mann-Whitney U Test (comparing two groups)

In [ ]:
print("="*60)
print("🔬 H4: Have prices increased over time?")
print("="*60)

years = sorted(df['year'].unique())
mid = years[len(years)//2]
early = df[df['year'] < mid]['close']
recent = df[df['year'] >= mid]['close']

h4_stat, h4_p = mannwhitneyu(early, recent)

print(f"\nEarly period ({years[0]}-{mid-1}): avg {early.mean():.2f}")
print(f"Recent period ({mid}-{years[-1]}): avg {recent.mean():.2f}")
print(f"Change: {((recent.mean() - early.mean()) / early.mean() * 100):.1f}%")
print(f"\nTest: Mann-Whitney U")
print(f"p-value: {h4_p:.2e}")
print()
if h4_p < 0.05:
    print("✅ SIGNIFICANT: Prices HAVE changed over time!")
else:
    print("❌ Not significant: No clear trend.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_temp = df.copy()
df_temp['period'] = df_temp['year'].apply(lambda x: 'Early' if x < mid else 'Recent')
sns.boxplot(data=df_temp, x='period', y='close', order=['Early', 'Recent'], 
            ax=axes[0], palette=['steelblue', 'coral'])
axes[0].set_title(f'H4: Early vs Recent Prices (p={h4_p:.2e})', fontweight='bold')

yearly = df.groupby('year')['close'].mean()
yearly.plot(ax=axes[1], marker='o', linewidth=2, color='steelblue')
z = np.polyfit(yearly.index.to_numpy(), yearly.to_numpy(), 1)
axes[1].plot(yearly.index, np.poly1d(z)(yearly.index), 'r--', label='Trend')
axes[1].axvline(x=mid, color='gray', linestyle=':', label='Split')
axes[1].set_title('Average Price by Year', fontweight='bold')
axes[1].legend()

result = '✅ SIGNIFICANT' if h4_p < 0.05 else '❌ NOT SIGNIFICANT'
axes[0].text(0.02, 0.98, result, transform=axes[0].transAxes, fontweight='bold', va='top',
             bbox=dict(boxstyle='round', facecolor='white'))

plt.tight_layout()
plt.savefig('../outputs/figures/h4_price_trend.png', dpi=300, bbox_inches='tight')
plt.show()

---
## Summary of All Tests

In [ ]:
print("="*70)
print("📋 HYPOTHESIS TESTING SUMMARY")
print("="*70)

results = [
    ['H1', 'Regional Differences', 'Kruskal-Wallis', f'{h1_p:.2e}', '✅ YES' if h1_p < 0.05 else '❌ NO'],
    ['H2', 'Volatility-Inflation', 'Spearman', f'{h2_p:.2e}', '✅ YES' if h2_p < 0.05 else '❌ NO'],
    ['H3', 'Seasonal Patterns', 'Kruskal-Wallis', f'{h3_p:.2e}', '✅ YES' if h3_p < 0.05 else '❌ NO'],
    ['H4', 'Price Trend', 'Mann-Whitney U', f'{h4_p:.2e}', '✅ YES' if h4_p < 0.05 else '❌ NO']
]

print(f"{'ID':<5} {'Question':<25} {'Test':<18} {'p-value':<12} {'Significant?'}")
print("-"*70)
for r in results:
    print(f"{r[0]:<5} {r[1]:<25} {r[2]:<18} {r[3]:<12} {r[4]}")

In [ ]:
# Save results
summary_df = pd.DataFrame(results, columns=['ID', 'Question', 'Test', 'p-value', 'Significant'])
summary_df.to_csv('../outputs/reports/hypothesis_test_results.csv', index=False)
print("💾 Saved: outputs/reports/hypothesis_test_results.csv")

---
## Conclusions

| Finding | Real-World Meaning |
|---------|-------------------|
| **H1: Countries differ** | Local factors matter - global solutions may not work everywhere |
| **H2: Volatility linked** | Stabilising prices could help control inflation |
| **H3: Seasonal patterns** | Food prices predictably higher in certain months |
| **H4: Prices increased** | Food affordability is a growing concern |

### Charts Created
- `h1_regional_inflation.png`
- `h2_volatility_inflation.png`
- `h3_seasonal_inflation.png`
- `h4_price_trend.png`

### Report Generated
- `hypothesis_test_results.csv`